In [1]:
pip install langchain langchain-community langchain-text-splitters chromadb faiss-cpu sentence-transformers pandas matplotlib seaborn

INFO: pip is looking at multiple versions of tokenizers to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of httpcore2 to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of httpcore2 to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of uvicorn[standard] to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of uvicorn[standard] to determine which version is compatible with other requirements. This could take a while.

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.42.1 requires protobuf<6,>=3.20, but you have protobuf 7.36.2 which is incompatible.

[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
import pandas as pd
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma, FAISS

C:\Users\prati\AppData\Local\Temp\ipykernel_9068\1949638909.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


1. Load Company Documents

In [4]:
loader = DirectoryLoader('./company_docs/', glob="*.txt", loader_cls=TextLoader)
documents = loader.load()
print(f"Loaded {len(documents)} document(s) successfully.")

Loaded 2 document(s) successfully.


2. Initialize Embedding Model

In [5]:
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

C:\Users\prati\AppData\Local\Temp\ipykernel_9068\2537902316.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\prati\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\prati\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

RAG Retrieval Performance Analysis (Chunk Size Experimentation)

In [6]:
def run_chunk_experiment(chunk_size, chunk_overlap, test_query):
    print(f"\n==============================================")
    print(f"Testing Chunk Size: {chunk_size} | Overlap: {chunk_overlap}")
    print(f"==============================================")
    
    # 1. Split text with specific chunk size
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    chunks = splitter.split_documents(documents)
    print(f"Total Chunks Created: {len(chunks)}")
    
    # 2. Store in ChromaDB Vector Database
    vector_db = Chroma.from_documents(
        chunks, 
        embedding_model, 
        collection_name=f"experiment_{chunk_size}"
    )
    
    # 3. Retrieve relevant chunks
    retriever = vector_db.as_retriever(search_kwargs={"k": 2})
    retrieved_docs = retriever.invoke(test_query)
    
    # 4. Display Results
    for i, doc in enumerate(retrieved_docs, 1):
        print(f"  [Retrieved Chunk {i}]: {doc.page_content.strip()}")
        
    return retrieved_docs

Run experiment with a test query

In [7]:
query = "What is the policy for home office equipment reimbursement?"
run_chunk_experiment(chunk_size=64, chunk_overlap=10, test_query=query)   # Small
run_chunk_experiment(chunk_size=250, chunk_overlap=30, test_query=query)  # Medium (Optimal)
run_chunk_experiment(chunk_size=600, chunk_overlap=50, test_query=query)  # Large


Testing Chunk Size: 64 | Overlap: 10
Total Chunks Created: 12
  [Retrieved Chunk 1]: stipend for home office equipment setup, including desks,
  [Retrieved Chunk 2]: desks, ergonomic chairs, and monitors. Internet reimbursement

Testing Chunk Size: 250 | Overlap: 30
Total Chunks Created: 6
  [Retrieved Chunk 1]: Remote Work and Equipment Policy:
  [Retrieved Chunk 2]: Full-time employees are permitted to work remotely up to 3 days a week. The company provides a one-time $500 stipend for home office equipment setup, including desks, ergonomic chairs, and monitors. Internet reimbursement is capped at $50 per month

Testing Chunk Size: 600 | Overlap: 50
Total Chunks Created: 2
  [Retrieved Chunk 1]: Remote Work and Equipment Policy:
Full-time employees are permitted to work remotely up to 3 days a week. The company provides a one-time $500 stipend for home office equipment setup, including desks, ergonomic chairs, and monitors. Internet reimbursement is capped at $50 per month upon submi

[Document(metadata={'source': 'company_docs\\remote_work.txt'}, page_content='Remote Work and Equipment Policy:\nFull-time employees are permitted to work remotely up to 3 days a week. The company provides a one-time $500 stipend for home office equipment setup, including desks, ergonomic chairs, and monitors. Internet reimbursement is capped at $50 per month upon submission of a valid bill.'),
 Document(metadata={'source': 'company_docs\\hr_policy.txt'}, page_content="Company Leave Policy:\nEmployees are entitled to 20 days of paid annual leave per year. Vacation requests must be submitted through the HR portal at least two weeks in advance. Sick leave allows up to 10 days annually with a doctor's note required for absences exceeding 3 consecutive days.")]

Vector Databases Integration (ChromaDB vs. FAISS)

In [8]:
# Optimal Chunks setup for production
optimal_splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=30)
optimal_chunks = optimal_splitter.split_documents(documents)

In [9]:
# 1. ChromaDB (Persistent / Local Collection)
chroma_db = Chroma.from_documents(optimal_chunks, embedding_model, collection_name="company_kb_chroma")
print(" ChromaDB vector store initialized.")

 ChromaDB vector store initialized.


In [10]:
#2. FAISS (Facebook AI Similarity Search)
faiss_db = FAISS.from_documents(optimal_chunks, embedding_model)
print(" FAISS vector store initialized.")

 FAISS vector store initialized.


Semantic Search & LLM Generation Pipeline

In [11]:
def rag_pipeline(user_query):
    # Retrieve top 2 most semantically similar chunks from FAISS or ChromaDB
    retriever = faiss_db.as_retriever(search_kwargs={"k": 2})
    relevant_docs = retriever.invoke(user_query)
    
    # Combine context
    context = "\n\n".join([doc.page_content for doc in relevant_docs])
    
    # Simulate LLM Prompt & Generation (Replace with OpenAI/ChatOpenAI if API key is available)
    ai_response = f"""=== RAG AI Response ===
User Query: {user_query}

Retrieved Context:
{context}

Generated Answer:
Based on the official company documents retrieved, here is the information you requested: 
{context}
======================="""
    
    return ai_response

Project Practical (Connect Company Docs ➔ Ask Questions ➔ Receive AI Answers)

In [12]:
print(" Company Documentation AI Assistant is Ready!\n")

while True:
    user_question = input("Ask a question about company policies (or type 'exit' to quit): ")
    if user_question.lower() == 'exit':
        print("Goodbye!")
        break
    
    if user_question.strip() == "":
        continue
        
    print("\nSearching documents and generating response...")
    response = rag_pipeline(user_question)
    print(response)
    print("\n" + "-"*50 + "\n")

 Company Documentation AI Assistant is Ready!


Searching documents and generating response...
=== RAG AI Response ===
User Query: cromDB

Retrieved Context:
for absences exceeding 3 consecutive days.

is capped at $50 per month upon submission of a valid bill.

Generated Answer:
Based on the official company documents retrieved, here is the information you requested: 
for absences exceeding 3 consecutive days.

is capped at $50 per month upon submission of a valid bill.

--------------------------------------------------


Searching documents and generating response...
=== RAG AI Response ===
User Query: what is RAG

Retrieved Context:
Employees are entitled to 20 days of paid annual leave per year. Vacation requests must be submitted through the HR portal at least two weeks in advance. Sick leave allows up to 10 days annually with a doctor's note required for absences exceeding 3

for absences exceeding 3 consecutive days.

Generated Answer:
Based on the official company documents re